# 02. Целевая переменная и feature engineering

Цель ноутбука — показать, какая переменная прогнозируется, зачем используется `log1p(gross)` и какие признаки создаются перед обучением модели.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ml_pipeline.feature_engineering import MovieFeatureEngineer

DATASET_PATH = PROJECT_ROOT / "data" / "output.csv"
df = pd.read_csv(DATASET_PATH)
df.head()

## Целевая переменная

В проекте прогнозируется `gross` — кассовая выручка фильма. Во время обучения используется логарифмическое преобразование:

`log_gross = log1p(gross)`

Это нужно, потому что выручка фильмов имеет сильно скошенное распределение: есть много фильмов со средней кассой и немного блокбастеров с очень большой кассой.

In [ ]:
df["log_gross"] = np.log1p(df["gross"])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df["gross"], bins=50, color="#ff8a5b", edgecolor="white")
axes[0].set_title("gross")
axes[0].set_xlabel("Кассовая выручка, USD")

axes[1].hist(df["log_gross"], bins=50, color="#2ec4b6", edgecolor="white")
axes[1].set_title("log1p(gross)")
axes[1].set_xlabel("log1p(gross)")

plt.tight_layout()
plt.show()

## Признаки production-модели

В production pipeline не используется `gross` как входной признак, потому что это целевая переменная. Поле `name/title` также не используется как ML-признак: название нового фильма плохо переносится на новые данные и используется только как metadata в интерфейсе.

In [ ]:
model_input_columns = [
    "rating", "genre", "year", "score", "votes", "director", "writer",
    "star", "country", "budget", "company", "runtime",
]

categorical_features = MovieFeatureEngineer.categorical_features
numeric_features = MovieFeatureEngineer.numeric_features
engineered_features = MovieFeatureEngineer.engineered_features

print("Входные колонки модели:")
display(pd.Series(model_input_columns, name="model_input_columns"))

print("Категориальные признаки:")
display(pd.Series(categorical_features, name="categorical_features"))

print("Числовые признаки после feature engineering:")
display(pd.Series(numeric_features, name="numeric_features"))

## Применение MovieFeatureEngineer

`MovieFeatureEngineer` — sklearn-compatible transformer. Он создаёт дополнительные признаки бюджета, голосов, длительности, года и оценки, а также сохраняет пороги по train-набору.

In [ ]:
X = df[model_input_columns].copy()
feature_engineer = MovieFeatureEngineer()
features = feature_engineer.fit_transform(X)

print("Размер до feature engineering:", X.shape)
print("Размер после feature engineering:", features.shape)
features.head()

In [ ]:
display(pd.DataFrame({
    "engineered_feature": engineered_features,
}))

## Связь признаков с целевой переменной

Для числовых признаков можно оценить корреляцию с `log_gross`. Это не заменяет обучение модели, но помогает объяснить значимость бюджета, голосов и длительности.

In [ ]:
numeric_for_corr = features[numeric_features].copy()
numeric_for_corr["log_gross"] = df["log_gross"].values

corr = numeric_for_corr.corr(numeric_only=True)["log_gross"].drop("log_gross").sort_values(ascending=False)
display(corr.to_frame("corr_with_log_gross"))

corr.head(12).sort_values().plot(kind="barh", figsize=(10, 6), color="#3c73ff")
plt.title("Корреляция числовых признаков с log_gross")
plt.xlabel("Correlation")
plt.tight_layout()
plt.show()

## Выводы

1. Целевая переменная проекта — `gross`.
2. Во время обучения используется `log1p(gross)`, а в production pipeline обратное преобразование выполняется автоматически через `expm1`.
3. `gross` и `name/title` не используются как входные признаки модели.
4. Основные инженерные признаки строятся вокруг бюджета, голосов, года, длительности и оценки.
5. Для будущей версии модели под российский рынок желательно переобучить модель без признаков `score` и `votes`, если прогноз делается до выхода фильма.

## Дополнение: feature schema из production artifacts

Production pipeline сохраняет `feature_schema.json`. Этот файл нужен, чтобы backend и будущие интеграции понимали, какие поля являются входными признаками, а какие являются metadata или target.

In [ ]:
import json

schema_path = PROJECT_ROOT / "artifacts" / "feature_schema.json"
with schema_path.open("r", encoding="utf-8") as file:
    feature_schema = json.load(file)

for key in ["model_input_columns", "categorical_features", "numeric_features", "engineered_features", "unused_columns", "target_column"]:
    print()
    print(f"{key}:")
    value = feature_schema[key]
    display(pd.Series(value if isinstance(value, list) else [value], name=key))


## Дополнение: почему не используется `name/title`

Название фильма может быть полезно для отображения результата пользователю, но оно плохо подходит как устойчивый ML-признак для новых фильмов: новые названия почти всегда неизвестны модели. Поэтому в production pipeline `title/name` используется как metadata, а не как обучающий признак.

In [ ]:
metadata_policy = pd.DataFrame([
    {"field": "title/name", "use_in_model": False, "reason": "metadata для отображения, слабая переносимость на новые фильмы"},
    {"field": "released", "use_in_model": False, "reason": "в текущей production-модели дата релиза не используется"},
    {"field": "gross", "use_in_model": False, "reason": "целевая переменная, недоступна до прогноза"},
    {"field": "budget", "use_in_model": True, "reason": "ключевой бизнес-признак фильма"},
])
metadata_policy